In [15]:
import json
import os
import sqlite3
import subprocess
from pymongo import MongoClient
import pandas as pd

In [16]:
# scrapy crawl ozon -o ozon_products.json

In [17]:
def get_default_gateway_ip():
    """Возвращает IP шлюза по умолчанию (адрес хоста Windows в WSL)"""
    try:
        result = subprocess.run(
            ["ip", "route", "show", "default"],
            capture_output=True, text=True
        )
        for line in result.stdout.splitlines():
            if "default via" in line:
                parts = line.split()
                for i, part in enumerate(parts):
                    if part == "via" and i+1 < len(parts):
                        return parts[i+1]
    except Exception:
        pass
    return None

def get_mongo_client():
    """
    Возвращает клиент MongoDB, подключённый к серверу на Windows.
    Сначала пытается через шлюз по умолчанию,
    затем через host.docker.internal, затем localhost.
    """
    # 1. Пробуем IP шлюза по умолчанию
    gw_ip = get_default_gateway_ip()
    if gw_ip:
        try:
            client = MongoClient(f'mongodb://{gw_ip}:27017/', serverSelectionTimeoutMS=2000)
            client.admin.command('ping')
            print(f"Подключено через шлюз {gw_ip}")
            return client
        except Exception:
            pass

    # 2. Пробуем host.docker.internal 
    try:
        client = MongoClient('mongodb://host.docker.internal:27017/', serverSelectionTimeoutMS=2000)
        client.admin.command('ping')
        print("Подключено через host.docker.internal")
        return client
    except Exception:
        pass

    # 3. Пробуем localhost 
    try:
        client = MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=2000)
        client.admin.command('ping')
        print("Подключено через localhost")
        return client
    except Exception:
        pass

    raise ConnectionError("Не удалось подключиться к MongoDB. Проверьте, что сервер запущен и доступен.")

def create_ozon_collection(db, collection_name='ozon'):
    """
    Создаёт коллекцию в MongoDB, если она не существует.
    В MongoDB коллекции создаются автоматически при вставке первого документа,
    но можно создать явно для контроля.
    """
    # Проверяем, существует ли коллекция
    if collection_name not in db.list_collection_names():
        db.create_collection(collection_name)
        print(f"Коллекция '{collection_name}' создана.")
    else:
        print(f"Коллекция '{collection_name}' уже существует.")

def drop_ozon_collection(db, collection_name='ozon'):
    """
    Удаляет коллекцию из MongoDB.
    """
    db.drop_collection(collection_name)
    print(f"Коллекция '{collection_name}' удалена.")

def fill_ozon_collection_from_json(db, json_path, collection_name='ozon'):
    """
    Заполняет коллекцию данными из JSON-файла.
    Ожидается, что JSON содержит список объектов с полями:
    image_href, name, cost, discount, rating, reviews, category_name, page_url, page_number.
    Каждый документ получит автоматически сгенерированное поле _id.
    """
    if not os.path.exists(json_path):
        print(f"Файл {json_path} не найден.")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        products = json.load(f)

    if not isinstance(products, list):
        print("JSON должен содержать список объектов.")
        return

    collection = db[collection_name]
    # Вставляем все документы (каждый документ — это словарь)
    result = collection.insert_many(products)
    print(f"Вставлено документов: {len(result.inserted_ids)}")

def get_high_discount_high_rating_mongo(db, collection_name='ozon'):
    """
    Выполняет запрос к коллекции MongoDB и возвращает записи с discount > 50 и rating > 4.5.
    """
    collection = db[collection_name]
    query = {
        'discount': {'$gt': 50},
        'rating': {'$gt': 4.5}
    }
    cursor = collection.find(query)
    data = list(cursor)
    if data:
        columns = list(data[0].keys())
    else:
        columns = []
    return data, columns

In [18]:
client = get_mongo_client()
db = client['ozon_db']
drop_ozon_collection(db, 'ozon_products_scrapy')
client.close()

Подключено через шлюз 172.25.208.1
Коллекция 'ozon_products_scrapy' удалена.


In [19]:
client = get_mongo_client()

db = client['ozon_db']

# Создаём коллекцию
create_ozon_collection(db, 'ozon_products_scrapy')

# Заполняем из JSON
fill_ozon_collection_from_json(db, '/mnt/c/Users/leous/.vscode/2_course/web_scrapping/scrapy/ozon_parser/output/ozon_2026-03-22T16-06-20+00-00.json', 'ozon_products_scrapy')
data, cols = get_high_discount_high_rating_mongo(db, 'ozon_products_scrapy')
client.close()
df = pd.DataFrame(data, columns=cols)
print(df.info())
df


Подключено через шлюз 172.25.208.1
Коллекция 'ozon_products_scrapy' создана.
Вставлено документов: 920
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   _id         530 non-null    object 
 1   image_href  530 non-null    object 
 2   name        530 non-null    object 
 3   cost        530 non-null    int64  
 4   discount    530 non-null    int64  
 5   rating      530 non-null    float64
 6   reviews     530 non-null    int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 29.1+ KB
None


,_id,image_href,name,cost,discount,rating,reviews
0,69c05559e5e657113cb59be4,https://www.ozon.ru/product/fleshka-4-gb-mr3-z...,"Флешка 4 гб. ""Мр3 Зарубежные 80-90-х - 2018 """,1024,74,4.9,146
1,69c05559e5e657113cb59be5,https://www.ozon.ru/product/depeche-mode-m-mem...,DEPECHE MODE M + Memento Mori (2025) DELUXE дв...,887,77,4.6,17
2,69c05559e5e657113cb59be7,https://www.ozon.ru/product/fleshka-8-gb-nashe...,"Флешка 8 гб. "" Наше радио 500 лучших """,1316,67,5.0,13
3,69c05559e5e657113cb59be8,https://www.ozon.ru/product/igor-talkov-luchsh...,Игорь Тальков - Лучшие Песни диск CD,263,67,4.8,165
4,69c05559e5e657113cb59be9,https://www.ozon.ru/product/fleshka-4-gb-legen...,Флешка 4 гб Легенды Ретро FM,986,75,4.9,27
...,...,...,...,...,...,...,...
525,69c05559e5e657113cb59f76,https://www.ozon.ru/product/zhidkiy-utyug-500m...,Жидкий утюг 500мл,311,82,4.7,1
526,69c05559e5e657113cb59f77,https://www.ozon.ru/product/plate-selteks-hlop...,"Платье Селтекс Хлопок, с сердечками",972,70,4.9,11
527,69c05559e5e657113cb59f78,https://www.ozon.ru/product/komplekt-odezhdy-m...,Комплект одежды MOOCIE,850,75,4.8,830
528,69c05559e5e657113cb59f79,https://www.ozon.ru/product/protect-rm-propitk...,Protect RM Пропитка водоотталкивающая,792,73,4.9,2


In [20]:
def upsert_ozon_mongo(db, collection_name, json_path):
    """
    Обновляет или вставляет документы в коллекцию MongoDB из JSON-файла.
    Ключ для обновления: image_href.
    """
    if not os.path.exists(json_path):
        print(f"Файл {json_path} не найден.")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        products = json.load(f)

    if not isinstance(products, list):
        print("JSON должен содержать список объектов.")
        return

    collection = db[collection_name]

    # Для ускорения можно создать индекс на поле image_href, если его нет
    collection.create_index('image_href', unique=True)

    inserted = 0
    updated = 0
    for item in products:
        # Используем image_href в качестве фильтра
        filter = {'image_href': item.get('image_href')}
        update = {'$set': item}  # обновляем все поля
        result = collection.update_one(filter, update, upsert=True)
        if result.upserted_id:
            inserted += 1
        elif result.modified_count:
            updated += 1

    print(f"Вставлено новых документов: {inserted}, обновлено: {updated}")

In [21]:
client = get_mongo_client()
db = client['ozon_db']
upsert_ozon_mongo(db, 'ozon_products_scrapy', "/mnt/c/Users/leous/.vscode/2_course/web_scrapping/scrapy/ozon_parser/output/ozon_2026-03-22T20-32-16+00-00.json")
data, cols = get_high_discount_high_rating_mongo(db, 'ozon_products_scrapy')
df = pd.DataFrame(data, columns=cols)
client.close()
print(df.info())
df

Подключено через шлюз 172.25.208.1
Вставлено новых документов: 1472, обновлено: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1418 entries, 0 to 1417
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   _id         1418 non-null   object 
 1   image_href  1418 non-null   object 
 2   name        1418 non-null   object 
 3   cost        1418 non-null   int64  
 4   discount    1418 non-null   int64  
 5   rating      1418 non-null   float64
 6   reviews     1418 non-null   int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 77.7+ KB
None


,_id,image_href,name,cost,discount,rating,reviews
0,69c05559e5e657113cb59be4,https://www.ozon.ru/product/fleshka-4-gb-mr3-z...,"Флешка 4 гб. ""Мр3 Зарубежные 80-90-х - 2018 """,1024,74,4.9,146
1,69c05559e5e657113cb59be5,https://www.ozon.ru/product/depeche-mode-m-mem...,DEPECHE MODE M + Memento Mori (2025) DELUXE дв...,887,77,4.6,17
2,69c05559e5e657113cb59be7,https://www.ozon.ru/product/fleshka-8-gb-nashe...,"Флешка 8 гб. "" Наше радио 500 лучших """,1316,67,5.0,13
3,69c05559e5e657113cb59be8,https://www.ozon.ru/product/igor-talkov-luchsh...,Игорь Тальков - Лучшие Песни диск CD,263,67,4.8,165
4,69c05559e5e657113cb59be9,https://www.ozon.ru/product/fleshka-4-gb-legen...,Флешка 4 гб Легенды Ретро FM,986,75,4.9,27
...,...,...,...,...,...,...,...
1413,69c0555a7ad1dfd190c3aeec,https://www.ozon.ru/product/bryuki-klassichesk...,Брюки классические SYJWY,910,82,4.8,9
1414,69c0555a7ad1dfd190c3aeed,https://www.ozon.ru/product/kupalnik-razdelnyy...,Купальник раздельный Love Skin женский / с выс...,1902,79,4.9,2
1415,69c0555a7ad1dfd190c3aeef,https://www.ozon.ru/product/komplekt-odezhdy-l...,Комплект одежды Lolapola с брюками оверсайз,1627,67,4.8,9
1416,69c0555a7ad1dfd190c3aef0,https://www.ozon.ru/product/komplekt-odezhdy-h...,Комплект одежды Happyfox для мужчин,2950,68,4.9,7
